# Question-Type Classification, Part 1: Word Embeddings & Text CNN


## Section 1: Fundamentals in RNNs

You need to **manually** implement a multi-timestep Recurrent Neural Network that can take an input as a 3D tensor `[batch_size, seq_len, input_size]` for a classification task.



In [62]:
import torch

We declare the relevant variables.

In [63]:
input_size = 5
seq_len = 4
batch_size = 8
hidden_size = 3
num_classes = 3

We create random inputs (i.e., `inputs`) and random labels (i.e., `random_labels`).

In [64]:
inputs = torch.randn(batch_size, seq_len, input_size)
random_labels = torch.randint(0, num_classes, (batch_size,))
print(inputs.shape)
print(random_labels)

torch.Size([8, 4, 5])
tensor([2, 0, 1, 0, 2, 2, 0, 1])


(1) In what follows, we need to declare the model parameters, which include the matrices $U$ (``[input_size, hidden_size]``), W (``[hidden_size, hidden_size]``), $V$ (``[hidden_size, num_classes]``) and the biases $b$ and $c$ for the hidden states and logits respectively.



In [65]:
# Initialize weight matrix for input-to-hidden transformation
U = torch.randn(input_size, hidden_size, requires_grad=True)

# Initialize weight matrix for hidden-to-hidden transformation
W = torch.randn(hidden_size, hidden_size, requires_grad=True)

# Initialize bias vector for hidden layer
b = torch.randn(hidden_size, requires_grad=True)

# Initialize weight matrix for hidden-to-output transformation
V = torch.randn(hidden_size, num_classes, requires_grad=True)

# Initialize bias vector for output layer
c = torch.randn(num_classes, requires_grad=True)

print("Model Parameters:")
print("U:", U)
print("W:", W)
print("V:", V)
print("b:", b)
print("c:", c)

Model Parameters:
U: tensor([[ 1.9946, -0.8209,  1.0061],
        [-1.0664, -0.4572,  0.0901],
        [ 0.4018, -1.1542,  0.1192],
        [-0.7348, -1.5864, -0.2671],
        [ 2.6874, -0.4633,  0.6639]], requires_grad=True)
W: tensor([[ 0.2383,  0.8305, -0.0838],
        [ 0.9523,  0.2016, -0.1714],
        [-0.8164, -0.6066, -0.4800]], requires_grad=True)
V: tensor([[-1.6705, -0.4716,  0.8130],
        [-0.4171, -2.3476,  1.9083],
        [-1.8304,  0.3223,  0.9229]], requires_grad=True)
b: tensor([ 1.1733,  0.3714, -2.3531], requires_grad=True)
c: tensor([-0.0534, -0.3743,  0.0549], requires_grad=True)


(2) Next you need to write the code to compute `hiddens` which is a 3D tensor of the shape ``[batch_size, seq_len, hidden_size]`` using the formula of the simple/standard RNN cells. You can freely modify the code below.



In [66]:
hiddens = torch.zeros(batch_size, seq_len, hidden_size)

# Initialize the starting hidden state with zeros
h_t = torch.zeros(batch_size, hidden_size)

# Iterate through each time step in the sequence
for t in range(seq_len):
    # Extract input vector at current time step
    x_t = inputs[:, t, :]
    # Compute new hidden state: h_t = tanh(x_t * U + h_t-1 * W + b)
    h_t = torch.tanh(torch.matmul(x_t, U) + torch.matmul(h_t, W) + b)
    # Store computed hidden state at position t
    hiddens[:, t, :] = h_t

print(hiddens)

tensor([[[ 0.0428,  0.7521, -0.9812],
         [-0.9639,  0.9936, -0.9976],
         [ 1.0000, -0.6238, -0.1974],
         [-0.9973,  0.9863, -0.9990]],

        [[-0.3650,  0.9942, -0.9898],
         [ 1.0000,  0.9872,  0.4011],
         [-0.9963,  0.9991, -0.9998],
         [ 1.0000, -0.7119, -0.6415]],

        [[ 0.9997, -0.9653, -0.7377],
         [ 0.9995,  0.9999, -0.9084],
         [ 0.8530,  0.9987, -0.9945],
         [ 0.9999,  1.0000, -0.9446]],

        [[ 0.8079, -0.6269, -0.9697],
         [-0.9699,  0.9580, -0.9966],
         [ 0.9927, -0.1821, -0.9440],
         [ 0.9657,  0.8308, -0.9342]],

        [[-0.0351,  0.1101, -0.9820],
         [ 1.0000,  0.8129, -0.6142],
         [-0.9942,  1.0000, -0.9997],
         [ 0.8543, -0.0014, -0.9532]],

        [[ 0.9913, -0.5334, -0.9988],
         [ 0.9998, -0.9797, -0.8479],
         [ 0.9998,  0.9973, -0.8168],
         [ 0.9934,  0.6805, -0.9430]],

        [[ 0.9919, -0.9317, -0.9506],
         [ 1.0000, -0.2871, -0.7248],


(3) In what follows, you need to write the code to compute the logits based on the last hidden state (``[batch_size, hidden_size]``) of hiddens.



In [67]:
logits = h_t@V + c
print(logits)

tensor([[ 3.0300, -2.5415,  0.2041],
        [-0.2528,  0.6185, -1.0825],
        [-0.4119, -3.4979,  1.9043],
        [-0.3032, -3.0811,  1.5632],
        [ 0.2647, -1.0810, -0.1330],
        [-0.2706, -2.7443,  1.2908],
        [-2.3080,  1.6556, -0.5197],
        [-0.2516, -3.4985,  1.8246]], grad_fn=<AddBackward0>)


(4) Write the code to compute the cross-entropy loss by comparing the logits to the labels. You can use PyTorch's built-in loss function.



In [68]:
loss = torch.nn.functional.cross_entropy(logits, random_labels)
print(loss)

tensor(2.8178, grad_fn=<NllLossBackward0>)


(5) Next, you need to do back-propagation to compute the gradients of the loss w.r.t. the model parameters. You can use PyTorch's built-in method to compute the gradients.



In [69]:
model_parameters = [U, W, b, V, c]
U_grad, W_grad, b_grad, V_grad, c_grad = torch.autograd.grad(
    outputs=loss,
    inputs=model_parameters,
    create_graph=False
)

print("Gradient for U:\n", U_grad)
print("Gradient for W:\n", W_grad)
print("Gradient for b:\n", b_grad)
print("Gradient for V:\n", V_grad)
print("Gradient for c:\n", c_grad)

Gradient for U:
 tensor([[ 0.0147,  0.0842,  0.4580],
        [-0.0706, -0.3418,  0.1713],
        [-0.1372, -0.4991, -0.2280],
        [ 0.0763,  0.3276,  0.2121],
        [ 0.0359,  0.0170,  0.4099]])
Gradient for W:
 tensor([[ 0.0663,  0.2919, -0.2334],
        [-0.0452, -0.2979,  0.2725],
        [ 0.0254,  0.2181, -0.3360]])
Gradient for b:
 tensor([-0.0284, -0.2258,  0.3378])
Gradient for V:
 tensor([[-0.3361, -0.0393,  0.3754],
        [ 0.2529, -0.4114,  0.1585],
        [-0.1302,  0.2328, -0.1027]])
Gradient for c:
 tensor([-0.0949, -0.0404,  0.1353])


(6) Finally, let assume that the learning rate $\eta = 0.1$, you need to write the code to **manually** update the new model parameters using the SGD manner.



In [70]:
lr = 0.1

with torch.no_grad():
    U = U - lr * U_grad
    W = W - lr * W_grad
    b = b - lr * b_grad
    V = V - lr * V_grad
    c = c - lr * c_grad

print("Updated model parameters:")
print("U:", U)
print("W:", W)
print("b:", b)
print("V:", V)
print("c:", c)

Updated model parameters:
U: tensor([[ 1.9931, -0.8293,  0.9603],
        [-1.0594, -0.4231,  0.0729],
        [ 0.4155, -1.1043,  0.1420],
        [-0.7424, -1.6192, -0.2883],
        [ 2.6838, -0.4650,  0.6230]])
W: tensor([[ 0.2317,  0.8013, -0.0605],
        [ 0.9568,  0.2314, -0.1986],
        [-0.8190, -0.6284, -0.4464]])
b: tensor([ 1.1762,  0.3940, -2.3869])
V: tensor([[-1.6369, -0.4676,  0.7755],
        [-0.4424, -2.3064,  1.8924],
        [-1.8174,  0.2991,  0.9332]])
c: tensor([-0.0439, -0.3703,  0.0414])


## Section 2: Deep Learning for Sequential Data

### <font color="#0b486b">Set random seeds</font>

We start with importing PyTorch and NumPy and setting random seeds for PyTorch and NumPy. You can use any seeds you prefer.

In [71]:
import os
import torch
import random
import requests
import pandas as pd
import numpy as np
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
from transformers import BertTokenizer
import os
from six.moves.urllib.request import urlretrieve
from sklearn import preprocessing
import matplotlib.pyplot as plt
plt.style.use('ggplot')

In [72]:
def seed_all(seed=1029):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
seed_all(seed=1234)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## <font color="#0b486b">Download and preprocess the data</font>

<div style="text-align: right"><font color="red; font-weight:bold"><span></div>

The dataset we use for this notebook is a question classification dataset for which the training set consists of $5,500$ questions belonging to 6 coarse question categories including:
- abbreviation (ABBR),
- entity (ENTY),
- description (DESC),
- human (HUM),
- location (LOC) and
- numeric (NUM).

In this notebook, we will utilize a subset of this dataset, containing $2,000$ questions for training and validation. We will use 80% of those 2000 questions for trainning and the rest for validation.


Preprocessing data is a crucial initial step in any machine learning or deep learning project. The *TextDataManager* class simplifies the process by providing functionalities to download and preprocess data specifically designed for the subsequent questions in this notebook. It is highly recommended to gain a comprehensive understanding of the class's functionality by **carefully reading** the content provided in the *TextDataManager* class before proceeding to answer the questions.

In [73]:
class DataManager:
    """
    This class manages and preprocesses a simple text dataset for a sentence classification task.

    Attributes:
        verbose (bool): Controls verbosity for printing information during data processing.
        max_sentence_len (int): The maximum length of a sentence in the dataset.
        str_questions (list): A list to store the string representations of the questions in the dataset.
        str_labels (list): A list to store the string representations of the labels in the dataset.
        numeral_labels (list): A list to store the numerical representations of the labels in the dataset.
        numeral_data (list): A list to store the numerical representations of the questions in the dataset.
        random_state (int): Seed value for random number generation to ensure reproducibility.
            Set this value to a specific integer to reproduce the same random sequence every time. Defaults to 6789.
        random (np.random.RandomState): Random number generator object initialized with the given random_state.
            It is used for various random operations in the class.

    Methods:
        maybe_download(dir_name, file_name, url, verbose=True):
            Downloads a file from a given URL if it does not exist in the specified directory.
            The directory and file are created if they do not exist.

        read_data(dir_name, file_names):
            Reads data from files in a directory, preprocesses it, and computes the maximum sentence length.
            Each file is expected to contain rows in the format "<label>:<question>".
            The labels and questions are stored as string representations.

        manipulate_data():
            Performs data manipulation by tokenizing, numericalizing, and padding the text data.
            The questions are tokenized and converted into numerical sequences using a tokenizer.
            The sequences are padded or truncated to the maximum sequence length.

        train_valid_test_split(train_ratio=0.9):
            Splits the data into training, validation, and test sets based on a given ratio.
            The data is randomly shuffled, and the specified ratio is used to determine the size of the training set.
            The string questions, numerical data, and numerical labels are split accordingly.
            TensorFlow `Dataset` objects are created for the training and validation sets.


    """

    def __init__(self, verbose=True, random_state=6789):
        self.verbose = verbose
        self.max_sentence_len = 0
        self.str_questions = list()
        self.str_labels = list()
        self.numeral_labels = list()
        self.maxlen = None
        self.numeral_data = list()
        self.random_state = random_state
        self.random = np.random.RandomState(random_state)

    @staticmethod
    def maybe_download(dir_name, file_name, url, verbose=True):
        if not os.path.exists(dir_name):
            os.mkdir(dir_name)
        if not os.path.exists(os.path.join(dir_name, file_name)):
            urlretrieve(url + file_name, os.path.join(dir_name, file_name))
        if verbose:
            print("Downloaded successfully {}".format(file_name))

    def read_data(self, dir_name, file_names):
        self.str_questions = list()
        self.str_labels = list()
        for file_name in file_names:
            file_path= os.path.join(dir_name, file_name)
            with open(file_path, "r", encoding="latin-1") as f:
                for row in f:
                    row_str = row.split(":")
                    label, question = row_str[0], row_str[1]
                    question = question.lower()
                    self.str_labels.append(label)
                    self.str_questions.append(question[0:-1])
                    if self.max_sentence_len < len(self.str_questions[-1]):
                        self.max_sentence_len = len(self.str_questions[-1])

        # turns labels into numbers
        le = preprocessing.LabelEncoder()
        le.fit(self.str_labels)
        self.numeral_labels = np.array(le.transform(self.str_labels))
        self.str_classes = le.classes_
        self.num_classes = len(self.str_classes)
        if self.verbose:
            print("\nSample questions and corresponding labels... \n")
            print(self.str_questions[0:5])
            print(self.str_labels[0:5])

    def manipulate_data(self):
        self.tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
        vocab = self.tokenizer.get_vocab()
        self.word2idx = {w: i for i, w in enumerate(vocab)}
        self.idx2word = {i:w for w,i in self.word2idx.items()}
        self.vocab_size = len(self.word2idx)

        token_ids = []
        num_seqs = []
        for text in self.str_questions:  # iterate over the list of text
          text_seqs = self.tokenizer.tokenize(str(text))  # tokenize each text individually
          # Convert tokens to IDs
          token_ids = self.tokenizer.convert_tokens_to_ids(text_seqs)
          # Convert token IDs to a tensor of indices using your word2idx mapping
          seq_tensor = torch.LongTensor(token_ids)
          num_seqs.append(seq_tensor)  # append the tensor for each sequence

        # Pad the sequences and create a tensor
        if num_seqs:
          self.numeral_data = pad_sequence(num_seqs, batch_first=True)  # Pads to max length of the sequences
          self.num_sentences, self.maxlen = self.numeral_data.shape

    def train_valid_test_split(self, train_ratio=0.8, test_ratio = 0.1):
        train_size = int(self.num_sentences*train_ratio) +1
        test_size = int(self.num_sentences*test_ratio) +1
        valid_size = self.num_sentences - (train_size + test_size)
        data_indices = list(range(self.num_sentences))
        random.shuffle(data_indices)
        self.train_str_questions = [self.str_questions[i] for i in data_indices[:train_size]]
        self.train_numeral_labels = self.numeral_labels[data_indices[:train_size]]
        train_set_data = self.numeral_data[data_indices[:train_size]]
        train_set_labels = self.numeral_labels[data_indices[:train_size]]
        train_set_labels = torch.from_numpy(train_set_labels)
        train_set = torch.utils.data.TensorDataset(train_set_data, train_set_labels)
        self.test_str_questions = [self.str_questions[i] for i in data_indices[-test_size:]]
        self.test_numeral_labels = self.numeral_labels[data_indices[-test_size:]]
        test_set_data = self.numeral_data[data_indices[-test_size:]]
        test_set_labels = self.numeral_labels[data_indices[-test_size:]]
        test_set_labels = torch.from_numpy(test_set_labels)
        test_set = torch.utils.data.TensorDataset(test_set_data, test_set_labels)
        self.valid_str_questions = [self.str_questions[i] for i in data_indices[train_size:-test_size]]
        self.valid_numeral_labels = self.numeral_labels[data_indices[train_size:-test_size]]
        valid_set_data = self.numeral_data[data_indices[train_size:-test_size]]
        valid_set_labels = self.numeral_labels[data_indices[train_size:-test_size]]
        valid_set_labels = torch.from_numpy(valid_set_labels)
        valid_set = torch.utils.data.TensorDataset(valid_set_data, valid_set_labels)
        self.train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
        self.test_loader = DataLoader(test_set, batch_size=64, shuffle=False)
        self.valid_loader = DataLoader(valid_set, batch_size=64, shuffle=False)

In [74]:
print('Loading data...')
DataManager.maybe_download("data", "train_2000.label", "http://cogcomp.org/Data/QA/QC/")

dm = DataManager()
dm.read_data("data/", ["train_2000.label"])

Loading data...
Downloaded successfully train_2000.label

Sample questions and corresponding labels... 

['manner how did serfdom develop in and then leave russia ?', 'cremat what films featured the character popeye doyle ?', "manner how can i find a list of celebrities ' real names ?", 'animal what fowl grabs the spotlight after the chinese year of the monkey ?', 'exp what is the full form of .com ?']
['DESC', 'ENTY', 'DESC', 'ENTY', 'ABBR']


In [75]:
dm.manipulate_data()
dm.train_valid_test_split(train_ratio=0.8, test_ratio = 0.1)

In [76]:
for x, y in dm.train_loader:
    print(x.shape, y.shape)
    break

torch.Size([64, 36]) torch.Size([64])


## <font color="#0b486b">Part 1: Using Word2Vect to transform texts to vectors </font>

<div style="text-align: right"><font color="red; font-weight:bold"><span></div>

In [77]:
!pip install gensim

In [78]:
import gensim.downloader as api
from gensim.models import Word2Vec
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score
import numpy as np

#### <font color="red">**1.1**</font>
**Write code to download the pretrained model *glove-wiki-gigaword-100*. Note that this model transforms a word in its dictionary to a $100$ dimensional vector.**

**Write code for the function *get_word_vector(word, model)* used to transform a word to a vector using the pretrained Word2Vect model *model*. Note that for a word not in the vocabulary of our *word2vect*, you need to return a vector $0$ with 100 dimensions.**



In [79]:
word2vect = api.load("glove-wiki-gigaword-100")

In [90]:
def get_word_vector(word, model):
    try:
        vector = model[word]
    except:
        vector = np.zeros(100)
    return vector

#### <font color="red">**1.2**</font>

**Write the code for the function `get_sentence_vector(sentence, important_score=None, model= None)`. Note that this function will transform a sentence to a 100-dimensional vector using the pretrained model *model*. In addition, the list *important_score* which has the same length as the *sentence* specifies the important scores of the words in the sentence. In your code, you first need to apply *softmax* function over *important_score* to obtain the important weight *important_weight* which forms a probability over the words of the sentence. Furthermore, the final vector of the sentence will be weighted sum of the individual vectors for words and the weights in *important_weight*.**
- $important\_weight = softmax(important\_score)$.
- $final\_vector= important\_weight[1]\times v[1] + important\_weight[2]\times v[2] + ...+ important\_weight[T]\times v[T]$ where $T$ is the length of the sentence and $v[i]$ is the vector representation of the $i-th$  word in this sentence.

**Note that if `important_score=None` is set by default, your function should return the average of all representation vectors corresponding to set `important_score=[1,1,...,1]`.**



In [91]:
def softmax(scores):
    """Apply softmax normalization to convert scores to probabilities."""
    exp_scores = np.exp(scores - np.max(scores))
    return exp_scores / np.sum(exp_scores)

def get_sentence_vector(sentence, important_score=None, model= None):
    # Validate that model parameter is provided
    if model is None:
        raise ValueError("Model can't be None")

    # Get embedding vectors for all words in the sentence
    embeddings = [get_word_vector(word, model) for word in sentence]

    # Default to uniform importance if not specified
    if important_score is None:
        important_score = [1] * len(sentence)

    # Convert importance scores to normalized weights using softmax
    important_weight = softmax(important_score)

    # Compute the weighted sum of embeddings
    feature_vector = np.sum([important_weight[i] * embeddings[i] for i in range(len(sentence))], axis=0)

    return feature_vector

#### <font color="red">**1.3**</font>

**Write code to transform questions in *dm.train_str_questions* and *dm.valid_str_questions* to feature vectors. Note that after running the following cells, you must have $X\_train$ and $X\_valid$ which are two NumPy arrays of the feature vectors and $y\_train$ and $y\_valid$ which are two arrays of numeric labels (Hint: *dm.train_numeral_labels* and *dm.valid_numeral_labels*). You can add more lines to the following cells if necessary. In addition, you should decide the *important_score* by yourself. For example, the 1st score is 1, the 2nd score is decayed by 0.9, the 3rd is decayed by 0.9, and so on.**



In [92]:
print("Transform training set to feature vectors...")
X_train = np.array([
    get_sentence_vector(
        sentence,
        [0.9 ** i for i in range(len(sentence))],
        word2vect
    )
    for sentence in dm.train_str_questions
])
y_train = dm.train_numeral_labels

Transform training set to feature vectors...


In [93]:
print("Transform validation set to feature vectors...")
X_valid = np.array([
    get_sentence_vector(
        sentence,
        [0.9 ** i for i in range(len(sentence))],
        word2vect
    )
    for sentence in dm.valid_str_questions
])
y_valid = dm.valid_numeral_labels

Transform validation set to feature vectors...


#### <font color="red">**1.4**</font>

**It is now to use *MinMaxScaler(feature_range=(-1,1))* in scikit-learn to scale both training and validation sets to the range $(-1,1)$.**



In [94]:
# Initialize scaler to map features to [-1, 1] range
scaler = MinMaxScaler(feature_range=(-1, 1))

# Fit scaler on training data and transform both sets
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

print(X_train_scaled.shape)
print(X_valid_scaled.shape)

(1601, 100)
(198, 100)


#### <font color="red">**1.5**</font>
**Train a Logistic Regression model on the training set and then evaluate on the validation set.** You can use any classification metrics in `sklearn` for evaluation.


In [95]:
from sklearn.linear_model import LogisticRegression

log_reg_model = LogisticRegression()
log_reg_model.fit(X_train_scaled, y_train)

LogisticRegression()

In [96]:
from sklearn import metrics

# Generate predictions
y_pred = log_reg_model.predict(X_valid_scaled)

# Calculate metrics
accuracy = metrics.accuracy_score(y_valid, y_pred)
precision = metrics.precision_score(y_valid, y_pred, average='weighted')
recall = metrics.recall_score(y_valid, y_pred, average='weighted')
f1_score = metrics.f1_score(y_valid, y_pred, average='weighted')

print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1 Score: {f1_score}")

Accuracy: 0.6161616161616161
Precision: 0.6184303791446649
Recall: 0.6161616161616161
F1 Score: 0.6165257674092768


We now declare the `BaseTrainer` class, which will be used later to train the subsequent deep learning models for text data.

In [97]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class BaseTrainer:
    def __init__(self, model, criterion, optimizer, train_loader, val_loader):
        self.model = model
        self.criterion = criterion  #the loss function
        self.optimizer = optimizer  #the optimizer
        self.train_loader = train_loader  #the train loader
        self.val_loader = val_loader  #the valid loader

    #the function to train the model in many epochs
    def fit(self, num_epochs):
        self.num_batches = len(self.train_loader)

        for epoch in range(num_epochs):
            print(f'Epoch {epoch + 1}/{num_epochs}')
            train_loss, train_accuracy = self.train_one_epoch()
            val_loss, val_accuracy = self.validate_one_epoch()
            print(
                f'{self.num_batches}/{self.num_batches} - train_loss: {train_loss:.4f} - train_accuracy: {train_accuracy*100:.4f}% \
                - val_loss: {val_loss:.4f} - val_accuracy: {val_accuracy*100:.4f}%')

    #train in one epoch, return the train_acc, train_loss
    def train_one_epoch(self):
        self.model.train()
        running_loss, correct, total = 0.0, 0, 0
        for i, data in enumerate(self.train_loader):
            inputs, labels = data
            inputs, labels = inputs.to(device), labels.to(device)
            self.optimizer.zero_grad()
            outputs = self.model(inputs)
            loss = self.criterion(outputs, labels)
            loss.backward()
            self.optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        train_accuracy = correct / total
        train_loss = running_loss / self.num_batches
        return train_loss, train_accuracy

    #evaluate on a loader and return the loss and accuracy
    def evaluate(self, loader):
        self.model.eval()
        loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for data in loader:
                inputs, labels = data
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = self.model(inputs)
                loss = self.criterion(outputs, labels)
                loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        accuracy = correct / total
        loss = loss / len(self.val_loader)
        return loss, accuracy

    #return the val_acc, val_loss, be called at the end of each epoch
    def validate_one_epoch(self):
      val_loss, val_accuracy = self.evaluate(self.val_loader)
      return val_loss, val_accuracy

## <font color="#0b486b">Part 2: Text CNN for sequence modeling and neural embedding </font>

<div style="text-align: right"><font color="red; font-weight:bold"><span></div>

**In what follows, you are required to complete the code for Text CNN for sentence classification. The paper of Text CNN can be found at this [link](https://www.aclweb.org/anthology/D14-1181.pdf). Here is the description of the Text CNN that you need to construct.**
- There are three attributes (properties or instance variables): *embed_size, state_size, data_manager*.
  - `embed_size`: the dimension of the vector space for which the words are embedded to using the embedding matrix.
  - `state_size`: the number of filters used in *Conv1D* (reference [here](https://pytorch.org/docs/stable/generated/torch.nn.Conv1d.html)).
  - `data_manager`: the data manager to store information of the dataset.
- The detail of the computational process is as follows:
  - Given input $x$, we embed $x$ using the embedding matrix to obtain an $3D$ tensor $[batch\_size, seq\_len, embed\_size]$ as $e$.
  - We feed $e$ to three *Conv1D* layers, each of which has $state\_size$ filters, activation= $relu$, and $kernel\_size= 3, 5, 7$ respectively to obtain $h1, h2, h3$. Note that each $h1, h2, h3$ is a 3D tensor with the shape $[batch\_size, state\_size, output\_size]$. Moreover, you need to apply *Conv1D* to the $seq\_len$ dimension.
  - We then apply *GlobalMaxPool1D()* (reference [here](https://pytorch.org/docs/stable/generated/torch.nn.functional.max_pool1d.html#torch.nn.functional.max_pool1d)) over $h1, h2, h3$ to obtain 2D tensors stored in $h1, h2, h3$ again.
  - We then concatenate three 2D tensors $h1, h2, h3$ to obtain $h$ with the shape $\left[batch\_size, 3\times state\_size\right]$. Note that you need to specify the axis to concatenate.
  - We finally build up one dense layer $\left[3\times state\_size, num\_classes\right]$  on the top of $h$ for classification.
  

In [98]:
import torch.nn as nn
import torch.nn.functional as F

#You can modify the code if you want but need to keep the skeleton
class TextCNN(torch.nn.Module):
    def __init__(self, embed_size= 128, state_size=16, data_manager=None):
        super().__init__()
        self.data_manager = data_manager
        self.embed_size = embed_size
        self.state_size = state_size
        #declare the necessary layers here
        self.embed = nn.Embedding(self.data_manager.vocab_size, self.embed_size)
        self.conv1d_1 = nn.Conv1d(in_channels=self.embed_size, out_channels=self.state_size, kernel_size=3)
        self.conv1d_2 = nn.Conv1d(in_channels=self.embed_size, out_channels=self.state_size, kernel_size=5)
        self.conv1d_3 = nn.Conv1d(in_channels=self.embed_size, out_channels=self.state_size, kernel_size=7)
        self.fc = nn.Linear(state_size*3, self.data_manager.num_classes)

    def forward(self, x):
        e = self.embed(x)
        #permute x before applying Conv1D
        e= e.permute(0,2,1)

        #applying Conv1D
        h1 = F.relu(self.conv1d_1(e))
        h2 = F.relu(self.conv1d_2(e))
        h3 = F.relu(self.conv1d_3(e))

        #apply GlobalMaxPool
        h1 = F.max_pool1d(h1, kernel_size=h1.shape[2]).squeeze(2)
        h2 = F.max_pool1d(h2, kernel_size=h2.shape[2]).squeeze(2)
        h3 = F.max_pool1d(h3, kernel_size=h3.shape[2]).squeeze(2)

        h = torch.cat((h1, h2, h3), dim=1)
        h = self.fc(h)
        return h

We declare `text_cnn` and train on several epochs (e.g., `50 epochs`).


In [99]:
text_cnn = TextCNN(data_manager=dm).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(text_cnn.parameters(), lr=0.001)
trainer = BaseTrainer(model=text_cnn, criterion=criterion, optimizer=optimizer, train_loader=dm.train_loader, val_loader=dm.valid_loader)
trainer.fit(num_epochs=50)

Epoch 1/50
26/26 - train_loss: 1.6139 - train_accuracy: 36.8520%                 - val_loss: 0.5991 - val_accuracy: 71.7172%
Epoch 2/50
26/26 - train_loss: 0.8245 - train_accuracy: 80.3248%                 - val_loss: 0.3028 - val_accuracy: 80.3030%
Epoch 3/50
26/26 - train_loss: 0.4867 - train_accuracy: 89.7564%                 - val_loss: 0.1956 - val_accuracy: 90.4040%
Epoch 4/50
26/26 - train_loss: 0.2666 - train_accuracy: 94.4410%                 - val_loss: 0.1611 - val_accuracy: 90.9091%
Epoch 5/50
26/26 - train_loss: 0.1737 - train_accuracy: 97.0643%                 - val_loss: 0.1311 - val_accuracy: 92.9293%
Epoch 6/50
26/26 - train_loss: 0.1166 - train_accuracy: 98.8757%                 - val_loss: 0.1156 - val_accuracy: 93.4343%
Epoch 7/50
26/26 - train_loss: 0.0937 - train_accuracy: 99.5628%                 - val_loss: 0.0813 - val_accuracy: 93.9394%
Epoch 8/50
26/26 - train_loss: 0.0773 - train_accuracy: 98.5009%                 - val_loss: 0.0964 - val_accuracy: 94.9495%


We evaluate the trained model on the testing set.

In [100]:
test_loss, test_acc = trainer.evaluate(dm.test_loader)
print(f'test_loss: {test_loss:.4f} - test_accuracy: {test_acc*100:.4f}%')

test_loss: 0.2677 - test_accuracy: 95.5224%
